<a href="https://colab.research.google.com/github/alaa-32/dashboard_djezzy/blob/main/cleaningpfe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

# Main project folder Drive
PROJECT_DIR = Path("/content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data")

# Input raw data
RAW_DIR = PROJECT_DIR / "raw_data"

# Output cleaned data Power BI
CLEAN_DIR = PROJECT_DIR / "clean_data"

# Output reports checking files
REPORT_DIR = PROJECT_DIR / "reports"

In [ ]:
#test check if they exist fel drive
expected_files = [
    "Activations Table.csv",
    "Cust Table.csv",
    "Package Table.csv",
    "POS Table .csv",
    "Reffil Table .csv",
    "Sales Table.csv"
]

for file_name in expected_files:
    file_path = RAW_DIR / file_name

    if file_path.exists():
        print("Found:", file_name)
    else:
        print("Missing:", file_name)

Found: Activations Table.csv
Found: Cust Table.csv
Found: Package Table.csv
Found: POS Table .csv
Found: Reffil Table .csv
Found: Sales Table.csv


# Import+read data (Extract part )

In [ ]:
sep=";" #to read it a seperator

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
def read_csv_safe(file_path):  #var

    #Reads data by trying different encodings.
    #The files are separated by semicolon ;.


    encodings = ["utf-8", "latin1", "cp1252"]

    for enc in encodings:
        try:
            df = pd.read_csv(
                file_path,
                sep=";",
                encoding=enc,
                dtype=str
            )

            print(f"Loaded: {file_path.name}")
            print(f"Encoding used: {enc}")
            print(f"Shape: {df.shape}")
            print("-" * 50)

            return df

        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not read file: {file_path}")

In [ ]:
activations_raw = read_csv_safe(RAW_DIR / "Activations Table.csv")
cust_raw = read_csv_safe(RAW_DIR / "Cust Table.csv")
package_raw = read_csv_safe(RAW_DIR / "Package Table.csv")
pos_raw = read_csv_safe(RAW_DIR / "POS Table .csv")
refill_raw = read_csv_safe(RAW_DIR / "Reffil Table .csv")
sales_raw = read_csv_safe(RAW_DIR / "Sales Table.csv")

Loaded: Activations Table.csv
Encoding used: utf-8
Shape: (10000, 5)
--------------------------------------------------
Loaded: Cust Table.csv
Encoding used: latin1
Shape: (10000, 11)
--------------------------------------------------
Loaded: Package Table.csv
Encoding used: latin1
Shape: (955, 5)
--------------------------------------------------
Loaded: POS Table .csv
Encoding used: utf-8
Shape: (10000, 12)
--------------------------------------------------
Loaded: Reffil Table .csv
Encoding used: utf-8
Shape: (10000, 11)
--------------------------------------------------
Loaded: Sales Table.csv
Encoding used: utf-8
Shape: (9595, 6)
--------------------------------------------------


In [ ]:
raw_tables = {
    "activations": activations_raw,
    "customers": cust_raw,
    "packages": package_raw,
    "pos": pos_raw,
    "refill": refill_raw,
    "sales": sales_raw
} #dic ndiro cleaning to all

In [ ]:
for table_name, df in raw_tables.items():
    print("=" * 60)
    print(table_name.upper())
    print("=" * 60)
    print("Rows and columns:", df.shape)
    print("Columns:")
    print(df.columns.tolist())
    print()
    #check

ACTIVATIONS
Rows and columns: (10000, 5)
Columns:
['subscriberNumber', 'adjustmentDate', 'Channel_code', 'transactioncode', 'adjustmentAmount']

CUSTOMERS
Rows and columns: (10000, 11)
Columns:
['MSISDN', 'POS_PRTNR_NMBR', 'DOB', 'GNDR', 'STRT', 'CITY', 'PRVC', 'CHNL', 'STS_DWH', 'ACT_DT', 'TER_DT']

PACKAGES
Rows and columns: (955, 5)
Columns:
['code', 'payment_types', 'name', 'periodic_unit', 'periodic_amount']

POS
Rows and columns: (10000, 12)
Columns:
['Code_PDV', 'Code_Agrument_', 'Region', 'wilaya_new', 'Commune', 'Etat_PDV', 'Created_Date', 'SIM_Partner_No', 'SIM_Parnter_Status', 'Auxiliary_SIM_No', 'Auxiliary_SIM_Status', 'POS_Org_Type']

REFILL
Rows and columns: (10000, 11)
Columns:
['Refill_Date', 'MSISDN', 'MSISDN_SRC', 'Refill_AMNT', 'RFL_SRC', 'RFL_CHNL', 'RFL_ACNT', 'RFL_TYPE', 'RFL_PRP', 'CITY', 'PRVC']

SALES
Rows and columns: (9595, 6)
Columns:
['MSISDN', 'Date_Vente', 'Date_Activation', 'Sales_Type', 'Line_Type', 'PDV_Subno']



# data understanding data profiling

In [ ]:
#general information about each table

for table_name, df in raw_tables.items():
    print("=" * 80)
    print(table_name.upper())
    print("=" * 80)

    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nFirst 5 rows:")
    display(df.head())

    print("\n")

ACTIVATIONS
Number of rows: 10000
Number of columns: 5

Columns:
['subscriberNumber', 'adjustmentDate', 'Channel_code', 'transactioncode', 'adjustmentAmount']

Duplicate rows: 0

First 5 rows:


,subscriberNumber,adjustmentDate,Channel_code,transactioncode,adjustmentAmount
0,865811048,18/07/2018,4,INTSPEEDDAY1PRE,100
1,861988207,02/07/2018,4,LIBERTYDAY50,50
2,865224119,19/07/2018,4,INTAMIGO1PRE,30
3,863807917,15/07/2018,1,Imtiyaz50PRE,50
4,851649153,19/08/2018,4,INTBSPEEDDAY1PRE,100




CUSTOMERS
Number of rows: 10000
Number of columns: 11

Columns:
['MSISDN', 'POS_PRTNR_NMBR', 'DOB', 'GNDR', 'STRT', 'CITY', 'PRVC', 'CHNL', 'STS_DWH', 'ACT_DT', 'TER_DT']

Duplicate rows: 0

First 5 rows:


,MSISDN,POS_PRTNR_NMBR,DOB,GNDR,STRT,CITY,PRVC,CHNL,STS_DWH,ACT_DT,TER_DT
0,853930004,869109274,01/01/1970,MALE,DUMMY_STREET,DUMMY_CITY,ADRAR,SMS888,TER,13/05/2019 21:18,11/09/2019 0:28
1,869570431,869859475,13/08/1984,MALE,CT AISSA BOUKARMA BT G,SKIKDA,SKIKDA,SMS888,TER,05/08/2018 17:44,16/05/2019 13:16
2,848212907,869109642,29/07/1993,FEMALE,435 LOTS G,1000,MASCARA,NaN,TER,25/12/2008 1:00,27/10/2020 5:31
3,842724151,869865733,01/03/1994,MALE,AIN EL MELH,AIN EL MELH,MSILA,SNOC,ACT,29/06/2023 8:55,01/01/3000 0:00
4,868441448,848712526,16/03/1995,MALE,BIR EL DJIR,BIR EL DJIR,ORAN,SNOC,TER,16/12/2021 19:21,09/05/2022 7:14




PACKAGES
Number of rows: 955
Number of columns: 5

Columns:
['code', 'payment_types', 'name', 'periodic_unit', 'periodic_amount']

Duplicate rows: 0

First 5 rows:


...

,Code_PDV,Code_Agrument_,Region,wilaya_new,Commune,Etat_PDV,Created_Date,SIM_Partner_No,SIM_Parnter_Status,Auxiliary_SIM_No,Auxiliary_SIM_Status,POS_Org_Type
0,28330047,0,Est,28-MSILA,33-OULED-DERRADJ,Active,28/06/2018,0,NaN,865916174,Active,POS
1,42180028,0,Center,42-TIPAZA,18-KHEMISTI,Terminated,28/06/2018,0,NaN,0,NaN,POS
2,28010940,0,Est,28-MSILA,01-MSILA2,Active,29/11/2018,0,NaN,862872634,Active,PoS
3,46110026,0,Ouest,46-AIN-TEMOUCHENT,11-EL AMIRIA,Terminated,28/06/2018,0,NaN,0,NaN,POS
4,35070011,1,Center,35-BOUMERDES,07-BORDJ-MENAIEL,Active,18/12/2015,867123290,Active,848711660,Active,Wholesaler




REFILL
Number of rows: 10000
Number of columns: 11

Columns:
['Refill_Date', 'MSISDN', 'MSISDN_SRC', 'Refill_AMNT', 'RFL_SRC', 'RFL_CHNL', 'RFL_ACNT', 'RFL_TYPE', 'RFL_PRP', 'CITY', 'PRVC']

Duplicate rows: 518

First 5 rows:


,Refill_Date,MSISDN,MSISDN_SRC,Refill_AMNT,RFL_SRC,RFL_CHNL,RFL_ACNT,RFL_TYPE,RFL_PRP,CITY,PRVC
0,19/03/2024,849425285,848715686,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,AIN EL HADJAR,SAIDA
1,19/03/2024,861894177,870203358,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,BOUATI MAHMOUD,GUELMA
2,19/03/2024,852294110,864708211,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CONSTANTINE,CONSTANTINE
3,19/03/2024,842274369,870268752,150,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,ORAN,ORAN
4,19/03/2024,843825102,848709760,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CHETTIA,CHLEF




SALES
Number of rows: 9595
Number of columns: 6

Columns:
['MSISDN', 'Date_Vente', 'Date_Activation', 'Sales_Type', 'Line_Type', 'PDV_Subno']

Duplicate rows: 0

First 5 rows:


,MSISDN,Date_Vente,Date_Activation,Sales_Type,Line_Type,PDV_Subno
0,846931477,29/06/2018,29/06/2018,SMS888,4G,869107410
1,846939913,30/06/2018,01/07/2018,SMS888,4G,867110788
2,846854401,30/06/2018,01/07/2018,SMS888,3G,869116085
3,846841682,30/06/2018,01/07/2018,SMS888,4G,867114282
4,846797123,30/06/2018,NaN,SMS888,4G,869112791


In [ ]:
#missing values
for table_name, df in raw_tables.items():
    print("=" * 80)
    print("MISSING VALUES IN:", table_name.upper())
    print("=" * 80)

    missing_report = pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percent": (df.isna().mean().values * 100).round(2)
    })

    display(missing_report)
    print("\n")

MISSING VALUES IN: ACTIVATIONS


,column,missing_count,missing_percent
0,subscriberNumber,0,0.00
1,adjustmentDate,0,0.00
2,Channel_code,56,0.56
3,transactioncode,0,0.00
4,adjustmentAmount,0,0.00




MISSING VALUES IN: CUSTOMERS


,column,missing_count,missing_percent
0,MSISDN,0,0.00
1,POS_PRTNR_NMBR,2361,23.61
2,DOB,37,0.37
3,GNDR,0,0.00
4,STRT,130,1.30
5,CITY,0,0.00
6,PRVC,4,0.04
7,CHNL,433,4.33
8,STS_DWH,0,0.00
9,ACT_DT,0,0.00




MISSING VALUES IN: PACKAGES


...

,column,missing_count,missing_percent
0,Code_PDV,0,0.00
1,Code_Agrument_,0,0.00
2,Region,0,0.00
3,wilaya_new,0,0.00
4,Commune,2,0.02
5,Etat_PDV,0,0.00
6,Created_Date,0,0.00
7,SIM_Partner_No,0,0.00
8,SIM_Parnter_Status,7584,75.84
9,Auxiliary_SIM_No,0,0.00




MISSING VALUES IN: REFILL


,column,missing_count,missing_percent
0,Refill_Date,515,5.15
1,MSISDN,515,5.15
2,MSISDN_SRC,515,5.15
3,Refill_AMNT,515,5.15
4,RFL_SRC,515,5.15
5,RFL_CHNL,515,5.15
6,RFL_ACNT,515,5.15
7,RFL_TYPE,515,5.15
8,RFL_PRP,515,5.15
9,CITY,515,5.15




MISSING VALUES IN: SALES


,column,missing_count,missing_percent
0,MSISDN,0,0.00
1,Date_Vente,0,0.00
2,Date_Activation,435,4.53
3,Sales_Type,0,0.00
4,Line_Type,1,0.01
5,PDV_Subno,0,0.00


In [ ]:
#Check data types it says object because we loaded the data dtype=str in padas it means text/string because we didnt want to lose the id if he read them as a number ou 9ader ynahi 0 win the first
for table_name, df in raw_tables.items():
    print("=" * 80)
    print("DATA TYPES IN:", table_name.upper())
    print("=" * 80)

    display(df.dtypes.to_frame("data_type"))
    print("\n")

DATA TYPES IN: ACTIVATIONS


,data_type
subscriberNumber,object
adjustmentDate,object
Channel_code,object
transactioncode,object
adjustmentAmount,object




DATA TYPES IN: CUSTOMERS


,data_type
MSISDN,object
POS_PRTNR_NMBR,object
DOB,object
GNDR,object
STRT,object
CITY,object
PRVC,object
CHNL,object
STS_DWH,object
ACT_DT,object




DATA TYPES IN: PACKAGES


...

,data_type
Code_PDV,object
Code_Agrument_,object
Region,object
wilaya_new,object
Commune,object
Etat_PDV,object
Created_Date,object
SIM_Partner_No,object
SIM_Parnter_Status,object
Auxiliary_SIM_No,object




DATA TYPES IN: REFILL


,data_type
Refill_Date,object
MSISDN,object
MSISDN_SRC,object
Refill_AMNT,object
RFL_SRC,object
RFL_CHNL,object
RFL_ACNT,object
RFL_TYPE,object
RFL_PRP,object
CITY,object




DATA TYPES IN: SALES


,data_type
MSISDN,object
Date_Vente,object
Date_Activation,object
Sales_Type,object
Line_Type,object
PDV_Subno,object


In [ ]:
#unique vaalues categorical column ( find problems fel writing POs pos so we clean them as one )
for table_name, df in raw_tables.items():
    print("=" * 80)
    print("UNIQUE VALUES SAMPLE IN:", table_name.upper())
    print("=" * 80)

    for col in df.columns:
        unique_count = df[col].nunique(dropna=True)

        # Show only columns with not too many unique values
        if unique_count <= 20:
            print(f"\nColumn: {col}")
            print(f"Unique values count: {unique_count}")
            print(df[col].dropna().unique()[:20])

    print("\n")

UNIQUE VALUES SAMPLE IN: ACTIVATIONS


UNIQUE VALUES SAMPLE IN: CUSTOMERS

Column: GNDR
Unique values count: 3
['MALE' 'FEMALE' 'NOTITLE']

Column: CHNL
Unique values count: 4
['SMS888' 'SNOC' 'POS' 'SFA']

Column: STS_DWH
Unique values count: 6
['TER' 'ACT' 'SUS' 'VAL' 'RTER' 'SSD']


UNIQUE VALUES SAMPLE IN: PACKAGES

Column: payment_types
Unique values count: 3
['prepaid' 'hybrid' 'postpaid']

Column: periodic_unit
Unique values count: 3
['days' 'hours' 'months']


UNIQUE VALUES SAMPLE IN: POS

Column: Code_Agrument_
Unique values count: 2
['0' '1']

Column: Region
Unique values count: 3
['Est' 'Center' 'Ouest']

Column: Etat_PDV
Unique values count: 3
['Active' 'Terminated' 'Initiated for Approval']

Column: SIM_Parnter_Status
Unique values count: 4
['Active' 'Terminated' 'Approval Initiated For Terminate'
 'Initiated for Approval']

Column: Auxiliary_SIM_Status
Unique values count: 3
['Active' 'Terminated' 'Approval Initiated For Termina']

Column: POS_Org_Type
Unique values count

In [ ]:
#Check empty rows
for table_name, df in raw_tables.items():
    empty_rows = df.isna().all(axis=1).sum()
    print(f"{table_name}: empty rows = {empty_rows}")

activations: empty rows = 0
customers: empty rows = 0
packages: empty rows = 0
pos: empty rows = 0
refill: empty rows = 515
sales: empty rows = 0


Create a raw data profiling report

In [ ]:
profiling_rows = []

for table_name, df in raw_tables.items():
    for col in df.columns:
        profiling_rows.append({
            "table": table_name,
            "column": col,
            "rows": df.shape[0],
            "missing_count": df[col].isna().sum(),
            "missing_percent": round(df[col].isna().mean() * 100, 2),
            "unique_values": df[col].nunique(dropna=True),
            "data_type": str(df[col].dtype)
        })

raw_profiling_report = pd.DataFrame(profiling_rows)

display(raw_profiling_report)

raw_profiling_report.to_csv(
    REPORT_DIR / "raw_data_profiling_report.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Raw profiling report exported to:")
print(REPORT_DIR / "raw_data_profiling_report.csv")

,table,column,rows,missing_count,missing_percent,unique_values,data_type
0,activations,subscriberNumber,10000,0,0.00,9983,object
1,activations,adjustmentDate,10000,0,0.00,2073,object
2,activations,Channel_code,10000,56,0.56,22,object
3,activations,transactioncode,10000,0,0.00,212,object
4,activations,adjustmentAmount,10000,0,0.00,42,object
5,customers,MSISDN,10000,0,0.00,9999,object
6,customers,POS_PRTNR_NMBR,10000,2361,23.61,4705,object
7,customers,DOB,10000,37,0.37,7083,object
8,customers,GNDR,10000,0,0.00,3,object
9,customers,STRT,10000,130,1.30,6961,object


Raw profiling report exported to:
/content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data/reports/raw_data_profiling_report.csv


# general cleaning functions (Transformation part)

In [ ]:
#clean column names
def clean_column_names(df):
    """
    Standardize column names:
    - remove spaces at the beginning/end
    - convert to lowercase
    - replace spaces with _
    - replace - with _
    - remove special characters
    """

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
    )

    return df

In [ ]:
#clean text values
def clean_text_value(x):
    """
    Clean text values:
    - remove extra spaces
    - remove non-breaking spaces
    - replace multiple spaces with one space
    - convert text to uppercase
    - convert empty values to NaN
    important fi power bi because he read them ad difrent one even if they are the same
    """

    if pd.isna(x):
        return np.nan

    x = str(x)
    x = x.replace("\xa0", " ")
    x = x.strip()
    x = re.sub(r"\s+", " ", x)

    if x == "" or x.lower() in ["nan", "none", "null"]:
        return np.nan

    return x.upper()

In [ ]:
#clean IDs
def clean_id(x):
    """
    Clean identifier columns:
    - keep IDs as text
    - remove spaces
    - remove .0 if it appears because of Excel
    - convert empty values to NaN
    """

    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(".0", "")

    if x == "" or x.lower() in ["nan", "none", "null"]:
        return np.nan

    return x

In [ ]:
#convert dates we did nat andd not remove because empty in our data may mean not active yet so no need to rush
def parse_date(series):
    """
    Convert a column to date format.
    dayfirst=True because dates are like 19/03/2024.
    errors='coerce' means invalid dates become NaT.
    """

    return pd.to_datetime(series, dayfirst=True, errors="coerce")

In [ ]:
#convert numeric columns some culums need to be number becaause in the first par i did turned them all to object now i return only the one i need to their origin
def clean_numeric(series):
    """
    Convert a column to numeric.
    Invalid values become NaN.
    """

    return pd.to_numeric(series, errors="coerce")

In [ ]:
#general cleaning dup mis empty function li elle caall the function
def standardize_table(df):
    """
    General cleaning applied to any table:
    - clean column names
    - remove completely empty rows
    - remove duplicate rows
    - clean all text values
    """

    df = df.copy()

    # Clean column names
    df = clean_column_names(df)

    # Remove rows where all columns are empty
    df = df.dropna(how="all")

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Clean text values in all columns
    for col in df.columns:
        df[col] = df[col].apply(clean_text_value)

    return df

In [ ]:
#to test
pos_test = standardize_table(pos_raw)

print("Original POS columns:")
print(pos_raw.columns.tolist())

print("\nCleaned POS columns:")
print(pos_test.columns.tolist())

print("\nUnique values before cleaning:")
print(pos_raw["POS_Org_Type"].dropna().unique())

print("\nUnique values after cleaning:")
print(pos_test["pos_org_type"].dropna().unique())

Original POS columns:
['Code_PDV', 'Code_Agrument_', 'Region', 'wilaya_new', 'Commune', 'Etat_PDV', 'Created_Date', 'SIM_Partner_No', 'SIM_Parnter_Status', 'Auxiliary_SIM_No', 'Auxiliary_SIM_Status', 'POS_Org_Type']

Cleaned POS columns:
['code_pdv', 'code_agrument_', 'region', 'wilaya_new', 'commune', 'etat_pdv', 'created_date', 'sim_partner_no', 'sim_parnter_status', 'auxiliary_sim_no', 'auxiliary_sim_status', 'pos_org_type']

Unique values before cleaning:
['POS' 'PoS' 'Wholesaler' 'B-POS']

Unique values after cleaning:
['POS' 'WHOLESALER' 'B-POS']


# customesrs table  
now we move to each table lkhaater m3a lowel derna general doka kayn change special for each table

In [ ]:
# generaal cleaning remove empty rows  duplicate rows standardize text values
cust = standardize_table(cust_raw)

print("Customers table after basic cleaning:")
print(cust.shape)
print(cust.columns.tolist())

display(cust.head())



Customers table after basic cleaning:
(10000, 11)
['msisdn', 'pos_prtnr_nmbr', 'dob', 'gndr', 'strt', 'city', 'prvc', 'chnl', 'sts_dwh', 'act_dt', 'ter_dt']


,msisdn,pos_prtnr_nmbr,dob,gndr,strt,city,prvc,chnl,sts_dwh,act_dt,ter_dt
0,853930004,869109274,01/01/1970,MALE,DUMMY_STREET,DUMMY_CITY,ADRAR,SMS888,TER,13/05/2019 21:18,11/09/2019 0:28
1,869570431,869859475,13/08/1984,MALE,CT AISSA BOUKARMA BT G,SKIKDA,SKIKDA,SMS888,TER,05/08/2018 17:44,16/05/2019 13:16
2,848212907,869109642,29/07/1993,FEMALE,435 LOTS G,1000,MASCARA,NaN,TER,25/12/2008 1:00,27/10/2020 5:31
3,842724151,869865733,01/03/1994,MALE,AIN EL MELH,AIN EL MELH,MSILA,SNOC,ACT,29/06/2023 8:55,01/01/3000 0:00
4,868441448,848712526,16/03/1995,MALE,BIR EL DJIR,BIR EL DJIR,ORAN,SNOC,TER,16/12/2021 19:21,09/05/2022 7:14


In [ ]:
# Rename columns for beter understanding when wee add it to power bi
cust = cust.rename(columns={
    "msisdn": "customer_msisdn",
    "pos_prtnr_nmbr": "pos_partner_number",
    "dob": "date_of_birth",
    "gndr": "gender",
    "strt": "street",
    "city": "customer_city",
    "prvc": "customer_wilaya",
    "chnl": "customer_channel",
    "sts_dwh": "customer_status",
    "act_dt": "customer_activation_date",
    "ter_dt": "customer_termination_date"
})

print(cust.columns.tolist())
display(cust.head())

['customer_msisdn', 'pos_partner_number', 'date_of_birth', 'gender', 'street', 'customer_city', 'customer_wilaya', 'customer_channel', 'customer_status', 'customer_activation_date', 'customer_termination_date']


,customer_msisdn,pos_partner_number,date_of_birth,gender,street,customer_city,customer_wilaya,customer_channel,customer_status,customer_activation_date,customer_termination_date
0,853930004,869109274,01/01/1970,MALE,DUMMY_STREET,DUMMY_CITY,ADRAR,SMS888,TER,13/05/2019 21:18,11/09/2019 0:28
1,869570431,869859475,13/08/1984,MALE,CT AISSA BOUKARMA BT G,SKIKDA,SKIKDA,SMS888,TER,05/08/2018 17:44,16/05/2019 13:16
2,848212907,869109642,29/07/1993,FEMALE,435 LOTS G,1000,MASCARA,NaN,TER,25/12/2008 1:00,27/10/2020 5:31
3,842724151,869865733,01/03/1994,MALE,AIN EL MELH,AIN EL MELH,MSILA,SNOC,ACT,29/06/2023 8:55,01/01/3000 0:00
4,868441448,848712526,16/03/1995,MALE,BIR EL DJIR,BIR EL DJIR,ORAN,SNOC,TER,16/12/2021 19:21,09/05/2022 7:14


In [ ]:
#clean id yb9aw text and we remove any unnecessary spaces or .0
cust["customer_msisdn"] = cust["customer_msisdn"].apply(clean_id)
cust["pos_partner_number"] = cust["pos_partner_number"].apply(clean_id)

display(cust[["customer_msisdn", "pos_partner_number"]].head())

,customer_msisdn,pos_partner_number
0,853930004,869109274
1,869570431,869859475
2,848212907,869109642
3,842724151,869865733
4,868441448,848712526


In [ ]:
#Convert date column
cust["date_of_birth"] = parse_date(cust["date_of_birth"])
cust["customer_activation_date"] = parse_date(cust["customer_activation_date"])
cust["customer_termination_date"] = parse_date(cust["customer_termination_date"])

display(cust[[
    "date_of_birth",
    "customer_activation_date",
    "customer_termination_date"
]].head())

print(cust[[
    "date_of_birth",
    "customer_activation_date",
    "customer_termination_date"
]].dtypes)

,date_of_birth,customer_activation_date,customer_termination_date
0,1970-01-01,2019-05-13 21:18:00,2019-09-11 00:28:00
1,1984-08-13,2018-08-05 17:44:00,2019-05-16 13:16:00
2,1993-07-29,2008-12-25 01:00:00,2020-10-27 05:31:00
3,1994-03-01,2023-06-29 08:55:00,NaT
4,1995-03-16,2021-12-16 19:21:00,2022-05-09 07:14:00


date_of_birth                datetime64[ns]
customer_activation_date     datetime64[ns]
customer_termination_date    datetime64[ns]
dtype: object


In [ ]:
#replace NOTITLE in gender because it is not a real gender we change it to unknow
cust["gender"] = cust["gender"].replace({
    "NOTITLE": "UNKNOWN"
})

print(cust["gender"].value_counts(dropna=False))

gender
MALE       8624
FEMALE     1347
UNKNOWN      29
Name: count, dtype: int64


In [ ]:
#Create active customer prepare for kpi later (transform ETL)
cust["is_customer_active"] = np.where(
    cust["customer_termination_date"].isna(),
    1,
    0
)

cust[["customer_status", "customer_termination_date", "is_customer_active"]].head()



,customer_status,customer_termination_date,is_customer_active
0,TER,2019-09-11 00:28:00,0
1,TER,2019-05-16 13:16:00,0
2,TER,2020-10-27 05:31:00,0
3,ACT,NaT,1
4,TER,2022-05-09 07:14:00,0


In [ ]:
#Create customer age and age group for dashboard
today = pd.Timestamp.today()

cust["customer_age"] = ((today - cust["date_of_birth"]).dt.days / 365.25).round(0)

# Remove unrealistic ages
cust.loc[
    (cust["customer_age"] < 10) | (cust["customer_age"] > 100),
    "customer_age"
] = np.nan

cust["customer_age_group"] = pd.cut(
    cust["customer_age"],
    bins=[0, 18, 25, 35, 45, 60, 100],
    labels=["<18", "18-25", "26-35", "36-45", "46-60", "60+"]
)

display(cust[["date_of_birth", "customer_age", "customer_age_group"]].head())

,date_of_birth,customer_age,customer_age_group
0,1970-01-01,56.0,46-60
1,1984-08-13,42.0,36-45
2,1993-07-29,33.0,26-35
3,1994-03-01,32.0,26-35
4,1995-03-16,31.0,26-35


In [ ]:
#check
print("Final Customers table shape:", cust.shape)

print("\nColumns:")
print(cust.columns.tolist())

print("\nMissing values:")
display(cust.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(cust.head())

Final Customers table shape: (10000, 14)

Columns:
['customer_msisdn', 'pos_partner_number', 'date_of_birth', 'gender', 'street', 'customer_city', 'customer_wilaya', 'customer_channel', 'customer_status', 'customer_activation_date', 'customer_termination_date', 'is_customer_active', 'customer_age', 'customer_age_group']

Missing values:


,missing_count
customer_msisdn,0
pos_partner_number,2361
date_of_birth,38
gender,0
street,130
customer_city,0
customer_wilaya,4
customer_channel,433
customer_status,0
customer_activation_date,0



Preview:


,customer_msisdn,pos_partner_number,date_of_birth,gender,street,customer_city,customer_wilaya,customer_channel,customer_status,customer_activation_date,customer_termination_date,is_customer_active,customer_age,customer_age_group
0,853930004,869109274,1970-01-01,MALE,DUMMY_STREET,DUMMY_CITY,ADRAR,SMS888,TER,2019-05-13 21:18:00,2019-09-11 00:28:00,0,56.0,46-60
1,869570431,869859475,1984-08-13,MALE,CT AISSA BOUKARMA BT G,SKIKDA,SKIKDA,SMS888,TER,2018-08-05 17:44:00,2019-05-16 13:16:00,0,42.0,36-45
2,848212907,869109642,1993-07-29,FEMALE,435 LOTS G,1000,MASCARA,NaN,TER,2008-12-25 01:00:00,2020-10-27 05:31:00,0,33.0,26-35
3,842724151,869865733,1994-03-01,MALE,AIN EL MELH,AIN EL MELH,MSILA,SNOC,ACT,2023-06-29 08:55:00,NaT,1,32.0,26-35
4,868441448,848712526,1995-03-16,MALE,BIR EL DJIR,BIR EL DJIR,ORAN,SNOC,TER,2021-12-16 19:21:00,2022-05-09 07:14:00,0,31.0,26-35


In [ ]:
#Create dim_customer clean data
dim_customer = cust[[
    "customer_msisdn",
    "pos_partner_number",
    "date_of_birth",
    "gender",
    "customer_age",
    "customer_age_group",
    "street",
    "customer_city",
    "customer_wilaya",
    "customer_channel",
    "customer_status",
    "customer_activation_date",
    "customer_termination_date",
    "is_customer_active"
]].drop_duplicates(subset=["customer_msisdn"])

print("dim_customer shape:", dim_customer.shape)
display(dim_customer.head())

dim_customer shape: (9999, 14)


,customer_msisdn,pos_partner_number,date_of_birth,gender,customer_age,customer_age_group,street,customer_city,customer_wilaya,customer_channel,customer_status,customer_activation_date,customer_termination_date,is_customer_active
0,853930004,869109274,1970-01-01,MALE,56.0,46-60,DUMMY_STREET,DUMMY_CITY,ADRAR,SMS888,TER,2019-05-13 21:18:00,2019-09-11 00:28:00,0
1,869570431,869859475,1984-08-13,MALE,42.0,36-45,CT AISSA BOUKARMA BT G,SKIKDA,SKIKDA,SMS888,TER,2018-08-05 17:44:00,2019-05-16 13:16:00,0
2,848212907,869109642,1993-07-29,FEMALE,33.0,26-35,435 LOTS G,1000,MASCARA,NaN,TER,2008-12-25 01:00:00,2020-10-27 05:31:00,0
3,842724151,869865733,1994-03-01,MALE,32.0,26-35,AIN EL MELH,AIN EL MELH,MSILA,SNOC,ACT,2023-06-29 08:55:00,NaT,1
4,868441448,848712526,1995-03-16,MALE,31.0,26-35,BIR EL DJIR,BIR EL DJIR,ORAN,SNOC,TER,2021-12-16 19:21:00,2022-05-09 07:14:00,0


#Clean the POS table

In [ ]:
#basic cleaning
pos = standardize_table(pos_raw)

print("POS table after basic cleaning:")
print(pos.shape)
print(pos.columns.tolist())

display(pos.head())

POS table after basic cleaning:
(10000, 12)
['code_pdv', 'code_agrument_', 'region', 'wilaya_new', 'commune', 'etat_pdv', 'created_date', 'sim_partner_no', 'sim_parnter_status', 'auxiliary_sim_no', 'auxiliary_sim_status', 'pos_org_type']


,code_pdv,code_agrument_,region,wilaya_new,commune,etat_pdv,created_date,sim_partner_no,sim_parnter_status,auxiliary_sim_no,auxiliary_sim_status,pos_org_type
0,28330047,0,EST,28-MSILA,33-OULED-DERRADJ,ACTIVE,28/06/2018,0,NaN,865916174,ACTIVE,POS
1,42180028,0,CENTER,42-TIPAZA,18-KHEMISTI,TERMINATED,28/06/2018,0,NaN,0,NaN,POS
2,28010940,0,EST,28-MSILA,01-MSILA2,ACTIVE,29/11/2018,0,NaN,862872634,ACTIVE,POS
3,46110026,0,OUEST,46-AIN-TEMOUCHENT,11-EL AMIRIA,TERMINATED,28/06/2018,0,NaN,0,NaN,POS
4,35070011,1,CENTER,35-BOUMERDES,07-BORDJ-MENAIEL,ACTIVE,18/12/2015,867123290,ACTIVE,848711660,ACTIVE,WHOLESALER


In [ ]:
#Rename POS column
pos = pos.rename(columns={
    "code_pdv": "pos_code",
    "code_agrument_": "agreement_code",
    "region": "region",
    "wilaya_new": "pos_wilaya_raw",
    "commune": "pos_commune_raw",
    "etat_pdv": "pos_status",
    "created_date": "pos_created_date",
    "sim_partner_no": "sim_partner_no",
    "sim_parnter_status": "sim_partner_status",
    "auxiliary_sim_no": "auxiliary_sim_no",
    "auxiliary_sim_status": "auxiliary_sim_status",
    "pos_org_type": "pos_org_type"
})

print(pos.columns.tolist())
display(pos.head())

['pos_code', 'agreement_code', 'region', 'pos_wilaya_raw', 'pos_commune_raw', 'pos_status', 'pos_created_date', 'sim_partner_no', 'sim_partner_status', 'auxiliary_sim_no', 'auxiliary_sim_status', 'pos_org_type']


,pos_code,agreement_code,region,pos_wilaya_raw,pos_commune_raw,pos_status,pos_created_date,sim_partner_no,sim_partner_status,auxiliary_sim_no,auxiliary_sim_status,pos_org_type
0,28330047,0,EST,28-MSILA,33-OULED-DERRADJ,ACTIVE,28/06/2018,0,NaN,865916174,ACTIVE,POS
1,42180028,0,CENTER,42-TIPAZA,18-KHEMISTI,TERMINATED,28/06/2018,0,NaN,0,NaN,POS
2,28010940,0,EST,28-MSILA,01-MSILA2,ACTIVE,29/11/2018,0,NaN,862872634,ACTIVE,POS
3,46110026,0,OUEST,46-AIN-TEMOUCHENT,11-EL AMIRIA,TERMINATED,28/06/2018,0,NaN,0,NaN,POS
4,35070011,1,CENTER,35-BOUMERDES,07-BORDJ-MENAIEL,ACTIVE,18/12/2015,867123290,ACTIVE,848711660,ACTIVE,WHOLESALER


In [ ]:
#clean id
id_columns = [
    "pos_code",
    "agreement_code",
    "sim_partner_no",
    "auxiliary_sim_no"
]

for col in id_columns:
    if col in pos.columns:
        pos[col] = pos[col].apply(clean_id)

display(pos[id_columns].head())

,pos_code,agreement_code,sim_partner_no,auxiliary_sim_no
0,28330047,0,0,865916174
1,42180028,0,0,0
2,28010940,0,0,862872634
3,46110026,0,0,0
4,35070011,1,867123290,848711660


In [ ]:
#Convert POS creation date
pos["pos_created_date"] = parse_date(pos["pos_created_date"])

display(pos[["pos_created_date"]].head())
print(pos["pos_created_date"].dtype)

,pos_created_date
0,2018-06-28
1,2018-06-28
2,2018-11-29
3,2018-06-28
4,2015-12-18


datetime64[ns]


In [ ]:
# our data 16-ALGER split so code alone 16 name alone alger (split wilaya)
pos["pos_wilaya_code"] = pos["pos_wilaya_raw"].str.extract(r"^(\d+)")
pos["pos_wilaya_name"] = pos["pos_wilaya_raw"].str.replace(r"^\d+-", "", regex=True)

# Create map-compatible wilaya ID
pos["wilaya_id"] = "DZ" + pos["pos_wilaya_code"].str.zfill(2)

display(pos[[
    "pos_wilaya_raw",
    "pos_wilaya_code",
    "pos_wilaya_name",
    "wilaya_id"
]].head())

,pos_wilaya_raw,pos_wilaya_code,pos_wilaya_name,wilaya_id
0,28-MSILA,28,MSILA,DZ28
1,42-TIPAZA,42,TIPAZA,DZ42
2,28-MSILA,28,MSILA,DZ28
3,46-AIN-TEMOUCHENT,46,AIN-TEMOUCHENT,DZ46
4,35-BOUMERDES,35,BOUMERDES,DZ35


In [ ]:
# split comune
pos["pos_commune_code"] = pos["pos_commune_raw"].str.extract(r"^(\d+)")
pos["pos_commune_name"] = pos["pos_commune_raw"].str.replace(r"^\d+-", "", regex=True)

display(pos[[
    "pos_commune_raw",
    "pos_commune_code",
    "pos_commune_name"
]].head())

,pos_commune_raw,pos_commune_code,pos_commune_name
0,33-OULED-DERRADJ,33,OULED-DERRADJ
1,18-KHEMISTI,18,KHEMISTI
2,01-MSILA2,01,MSILA2
3,11-EL AMIRIA,11,EL AMIRIA
4,07-BORDJ-MENAIEL,07,BORDJ-MENAIEL


In [ ]:
# active POS flag
pos["is_pos_active"] = np.where(
    pos["pos_status"] == "ACTIVE",
    1,
    0
)

display(pos[["pos_status", "is_pos_active"]].head())
print(pos["pos_status"].value_counts(dropna=False))

,pos_status,is_pos_active
0,ACTIVE,1
1,TERMINATED,0
2,ACTIVE,1
3,TERMINATED,0
4,ACTIVE,1


pos_status
ACTIVE                    6044
TERMINATED                3937
INITIATED FOR APPROVAL      19
Name: count, dtype: int64


In [ ]:
#test type
print("POS organization types:")
print(pos["pos_org_type"].value_counts(dropna=False))

POS organization types:
pos_org_type
POS           7522
B-POS         2339
WHOLESALER     139
Name: count, dtype: int64


In [ ]:
print("Final POS table shape:", pos.shape)

print("\nColumns:")
print(pos.columns.tolist())

print("\nMissing values:")
display(pos.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(pos.head())

Final POS table shape: (10000, 18)

Columns:
['pos_code', 'agreement_code', 'region', 'pos_wilaya_raw', 'pos_commune_raw', 'pos_status', 'pos_created_date', 'sim_partner_no', 'sim_partner_status', 'auxiliary_sim_no', 'auxiliary_sim_status', 'pos_org_type', 'pos_wilaya_code', 'pos_wilaya_name', 'wilaya_id', 'pos_commune_code', 'pos_commune_name', 'is_pos_active']

Missing values:


,missing_count
pos_code,0
agreement_code,0
region,0
pos_wilaya_raw,0
pos_commune_raw,2
pos_status,0
pos_created_date,0
sim_partner_no,0
sim_partner_status,7584
auxiliary_sim_no,0



Preview:


,pos_code,agreement_code,region,pos_wilaya_raw,pos_commune_raw,pos_status,pos_created_date,sim_partner_no,sim_partner_status,auxiliary_sim_no,auxiliary_sim_status,pos_org_type,pos_wilaya_code,pos_wilaya_name,wilaya_id,pos_commune_code,pos_commune_name,is_pos_active
0,28330047,0,EST,28-MSILA,33-OULED-DERRADJ,ACTIVE,2018-06-28,0,NaN,865916174,ACTIVE,POS,28,MSILA,DZ28,33,OULED-DERRADJ,1
1,42180028,0,CENTER,42-TIPAZA,18-KHEMISTI,TERMINATED,2018-06-28,0,NaN,0,NaN,POS,42,TIPAZA,DZ42,18,KHEMISTI,0
2,28010940,0,EST,28-MSILA,01-MSILA2,ACTIVE,2018-11-29,0,NaN,862872634,ACTIVE,POS,28,MSILA,DZ28,01,MSILA2,1
3,46110026,0,OUEST,46-AIN-TEMOUCHENT,11-EL AMIRIA,TERMINATED,2018-06-28,0,NaN,0,NaN,POS,46,AIN-TEMOUCHENT,DZ46,11,EL AMIRIA,0
4,35070011,1,CENTER,35-BOUMERDES,07-BORDJ-MENAIEL,ACTIVE,2015-12-18,867123290,ACTIVE,848711660,ACTIVE,WHOLESALER,35,BOUMERDES,DZ35,07,BORDJ-MENAIEL,1


In [ ]:
#creat dim pos
dim_pos = pos[[
    "pos_code",
    "agreement_code",
    "region",
    "pos_wilaya_code",
    "wilaya_id",
    "pos_wilaya_name",
    "pos_commune_code",
    "pos_commune_name",
    "pos_status",
    "pos_created_date",
    "sim_partner_no",
    "sim_partner_status",
    "auxiliary_sim_no",
    "auxiliary_sim_status",
    "pos_org_type",
    "is_pos_active"
]].drop_duplicates(subset=["pos_code"])

# Clean the Package table

In [ ]:
#general cleaning
package = standardize_table(package_raw)

print("Package table after basic cleaning:")
print(package.shape)
print(package.columns.tolist())

display(package.head())

Package table after basic cleaning:
(955, 5)
['code', 'payment_types', 'name', 'periodic_unit', 'periodic_amount']


,code,payment_types,name,periodic_unit,periodic_amount
0,INTBAMIGO1PRE,PREPAID,B2B AMIGO 50MO PREP 24 HOURS,DAYS,1
1,ACQSFA78000,HYBRID,ACQUISITION SFA PURCHASING PRICE 78000 DA,DAYS,1
2,HAYLADAYPREP,PREPAID,HAYLA 100 (PREP) DAIY,DAYS,1
3,IMTIYAZSURPRISE99IS240PRE,PREPAID,VOICE 240 M DJEZZY PREP DAILY,DAYS,1
4,PREPAIDDJEZZYINTERNET2000,PREPAID,PREPAID DJEZZY INTERNET 2000,HOURS,720


In [ ]:
#rename column
package = package.rename(columns={
    "code": "package_code",
    "payment_types": "payment_type",
    "name": "package_name",
    "periodic_unit": "periodic_unit",
    "periodic_amount": "periodic_amount"
})

print(package.columns.tolist())
display(package.head())

['package_code', 'payment_type', 'package_name', 'periodic_unit', 'periodic_amount']


,package_code,payment_type,package_name,periodic_unit,periodic_amount
0,INTBAMIGO1PRE,PREPAID,B2B AMIGO 50MO PREP 24 HOURS,DAYS,1
1,ACQSFA78000,HYBRID,ACQUISITION SFA PURCHASING PRICE 78000 DA,DAYS,1
2,HAYLADAYPREP,PREPAID,HAYLA 100 (PREP) DAIY,DAYS,1
3,IMTIYAZSURPRISE99IS240PRE,PREPAID,VOICE 240 M DJEZZY PREP DAILY,DAYS,1
4,PREPAIDDJEZZYINTERNET2000,PREPAID,PREPAID DJEZZY INTERNET 2000,HOURS,720


In [ ]:
#Clean package code stay text
package["package_code"] = package["package_code"].apply(clean_id)

display(package[["package_code", "package_name"]].head())

,package_code,package_name
0,INTBAMIGO1PRE,B2B AMIGO 50MO PREP 24 HOURS
1,ACQSFA78000,ACQUISITION SFA PURCHASING PRICE 78000 DA
2,HAYLADAYPREP,HAYLA 100 (PREP) DAIY
3,IMTIYAZSURPRISE99IS240PRE,VOICE 240 M DJEZZY PREP DAILY
4,PREPAIDDJEZZYINTERNET2000,PREPAID DJEZZY INTERNET 2000


**first :**
we Convert periodic amount to numeric because periodic_amount is a number. Example:
7 DAYS
24 HOURS
1 MONTH
So we convert only the amount part to numeric.


---


**second :**
Convert all validity durations to days Because the original table has different unit and we cant compare

---


**last :**
Create validity category
validity_days is numeric and useful for calculations.
validity_category is text and useful for Power BI visuals.
validity_days	validity_category


In [ ]:
#convert periodic amount to numeric
package["periodic_amount"] = clean_numeric(package["periodic_amount"])

display(package[["package_name", "periodic_unit", "periodic_amount"]].head())
print(package["periodic_amount"].dtype)

,package_name,periodic_unit,periodic_amount
0,B2B AMIGO 50MO PREP 24 HOURS,DAYS,1
1,ACQUISITION SFA PURCHASING PRICE 78000 DA,DAYS,1
2,HAYLA 100 (PREP) DAIY,DAYS,1
3,VOICE 240 M DJEZZY PREP DAILY,DAYS,1
4,PREPAID DJEZZY INTERNET 2000,HOURS,720


int64


In [ ]:
#Convert all validity durations to days
def convert_validity_to_days(row):
    amount = row["periodic_amount"]
    unit = row["periodic_unit"]

    if pd.isna(amount) or pd.isna(unit):
        return np.nan

    if unit == "HOURS":
        return amount / 24

    elif unit == "DAYS":
        return amount

    elif unit == "MONTHS":
        return amount * 30

    else:
        return np.nan


package["validity_days"] = package.apply(convert_validity_to_days, axis=1)

display(package[[
    "package_name",
    "periodic_amount",
    "periodic_unit",
    "validity_days"
]].head())

,package_name,periodic_amount,periodic_unit,validity_days
0,B2B AMIGO 50MO PREP 24 HOURS,1,DAYS,1.0
1,ACQUISITION SFA PURCHASING PRICE 78000 DA,1,DAYS,1.0
2,HAYLA 100 (PREP) DAIY,1,DAYS,1.0
3,VOICE 240 M DJEZZY PREP DAILY,1,DAYS,1.0
4,PREPAID DJEZZY INTERNET 2000,720,HOURS,30.0


In [ ]:
#Create validity category
def classify_validity(days):
    if pd.isna(days):
        return np.nan

    if days <= 1:
        return "1 DAY OR LESS"
    elif days <= 7:
        return "1 WEEK"
    elif days <= 14:
        return "2 WEEKS"
    elif days <= 30:
        return "1 MONTH"
    elif days <= 90:
        return "3 MONTHS"
    elif days <= 180:
        return "6 MONTHS"
    elif days <= 365:
        return "1 YEAR"
    else:
        return "MORE THAN 1 YEAR"


package["validity_category"] = package["validity_days"].apply(classify_validity)

display(package[[
    "package_name",
    "validity_days",
    "validity_category"
]].head())

,package_name,validity_days,validity_category
0,B2B AMIGO 50MO PREP 24 HOURS,1.0,1 DAY OR LESS
1,ACQUISITION SFA PURCHASING PRICE 78000 DA,1.0,1 DAY OR LESS
2,HAYLA 100 (PREP) DAIY,1.0,1 DAY OR LESS
3,VOICE 240 M DJEZZY PREP DAILY,1.0,1 DAY OR LESS
4,PREPAID DJEZZY INTERNET 2000,30.0,1 MONTH


In [ ]:
#Check package categories
print("Payment types:")
print(package["payment_type"].value_counts(dropna=False))

print("\nPeriodic units:")
print(package["periodic_unit"].value_counts(dropna=False))

print("\nValidity categories:")
print(package["validity_category"].value_counts(dropna=False))

Payment types:
payment_type
PREPAID     654
HYBRID      202
POSTPAID     99
Name: count, dtype: int64

Periodic units:
periodic_unit
DAYS      514
HOURS     424
MONTHS     17
Name: count, dtype: int64

Validity categories:
validity_category
1 MONTH             525
1 DAY OR LESS       222
1 WEEK              139
3 MONTHS             46
6 MONTHS              9
1 YEAR                8
MORE THAN 1 YEAR      3
2 WEEKS               3
Name: count, dtype: int64


In [ ]:
#final check
print("Final Package table shape:", package.shape)

print("\nColumns:")
print(package.columns.tolist())

print("\nMissing values:")
display(package.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(package.head())

Final Package table shape: (955, 7)

Columns:
['package_code', 'payment_type', 'package_name', 'periodic_unit', 'periodic_amount', 'validity_days', 'validity_category']

Missing values:


,missing_count
package_code,0
payment_type,0
package_name,0
periodic_unit,0
periodic_amount,0
validity_days,0
validity_category,0



Preview:


,package_code,payment_type,package_name,periodic_unit,periodic_amount,validity_days,validity_category
0,INTBAMIGO1PRE,PREPAID,B2B AMIGO 50MO PREP 24 HOURS,DAYS,1,1.0,1 DAY OR LESS
1,ACQSFA78000,HYBRID,ACQUISITION SFA PURCHASING PRICE 78000 DA,DAYS,1,1.0,1 DAY OR LESS
2,HAYLADAYPREP,PREPAID,HAYLA 100 (PREP) DAIY,DAYS,1,1.0,1 DAY OR LESS
3,IMTIYAZSURPRISE99IS240PRE,PREPAID,VOICE 240 M DJEZZY PREP DAILY,DAYS,1,1.0,1 DAY OR LESS
4,PREPAIDDJEZZYINTERNET2000,PREPAID,PREPAID DJEZZY INTERNET 2000,HOURS,720,30.0,1 MONTH


In [ ]:
# creat dim package
dim_package = package[[
    "package_code",
    "payment_type",
    "package_name",
    "periodic_unit",
    "periodic_amount",
    "validity_days",
    "validity_category"
]].drop_duplicates(subset=["package_code"])

print("dim_package shape:", dim_package.shape)
display(dim_package.head())

dim_package shape: (955, 7)


,package_code,payment_type,package_name,periodic_unit,periodic_amount,validity_days,validity_category
0,INTBAMIGO1PRE,PREPAID,B2B AMIGO 50MO PREP 24 HOURS,DAYS,1,1.0,1 DAY OR LESS
1,ACQSFA78000,HYBRID,ACQUISITION SFA PURCHASING PRICE 78000 DA,DAYS,1,1.0,1 DAY OR LESS
2,HAYLADAYPREP,PREPAID,HAYLA 100 (PREP) DAIY,DAYS,1,1.0,1 DAY OR LESS
3,IMTIYAZSURPRISE99IS240PRE,PREPAID,VOICE 240 M DJEZZY PREP DAILY,DAYS,1,1.0,1 DAY OR LESS
4,PREPAIDDJEZZYINTERNET2000,PREPAID,PREPAID DJEZZY INTERNET 2000,HOURS,720,30.0,1 MONTH


# Clean the Sales table

In [ ]:
#basic clean
sales = standardize_table(sales_raw)

print("Sales table after basic cleaning:")
print(sales.shape)
print(sales.columns.tolist())

display(sales.head())

#Rename columns
sales = sales.rename(columns={
    "msisdn": "customer_msisdn",
    "date_vente": "sale_date",
    "date_activation": "line_activation_date",
    "sales_type": "sales_type",
    "line_type": "line_type",
    "pdv_subno": "pdv_subno"
})

print(sales.columns.tolist())
display(sales.head())

Sales table after basic cleaning:
(9595, 6)
['msisdn', 'date_vente', 'date_activation', 'sales_type', 'line_type', 'pdv_subno']


,msisdn,date_vente,date_activation,sales_type,line_type,pdv_subno
0,846931477,29/06/2018,29/06/2018,SMS888,4G,869107410
1,846939913,30/06/2018,01/07/2018,SMS888,4G,867110788
2,846854401,30/06/2018,01/07/2018,SMS888,3G,869116085
3,846841682,30/06/2018,01/07/2018,SMS888,4G,867114282
4,846797123,30/06/2018,NaN,SMS888,4G,869112791


['customer_msisdn', 'sale_date', 'line_activation_date', 'sales_type', 'line_type', 'pdv_subno']


,customer_msisdn,sale_date,line_activation_date,sales_type,line_type,pdv_subno
0,846931477,29/06/2018,29/06/2018,SMS888,4G,869107410
1,846939913,30/06/2018,01/07/2018,SMS888,4G,867110788
2,846854401,30/06/2018,01/07/2018,SMS888,3G,869116085
3,846841682,30/06/2018,01/07/2018,SMS888,4G,867114282
4,846797123,30/06/2018,NaN,SMS888,4G,869112791


In [ ]:
#Clean ID columns
sales["customer_msisdn"] = sales["customer_msisdn"].apply(clean_id)
sales["pdv_subno"] = sales["pdv_subno"].apply(clean_id)

display(sales[["customer_msisdn", "pdv_subno"]].head())

#covert dates
sales["sale_date"] = parse_date(sales["sale_date"])
sales["line_activation_date"] = parse_date(sales["line_activation_date"])

display(sales[["sale_date", "line_activation_date"]].head())

print(sales[["sale_date", "line_activation_date"]].dtypes)

,customer_msisdn,pdv_subno
0,846931477,869107410
1,846939913,867110788
2,846854401,869116085
3,846841682,867114282
4,846797123,869112791


,sale_date,line_activation_date
0,2018-06-29,2018-06-29
1,2018-06-30,2018-07-01
2,2018-06-30,2018-07-01
3,2018-06-30,2018-07-01
4,2018-06-30,NaT


sale_date               datetime64[ns]
line_activation_date    datetime64[ns]
dtype: object


In [ ]:
#Create activation flag to know the sold line was activated or not.
sales["is_line_activated"] = np.where(
    sales["line_activation_date"].notna(),
    1,
    0
)

display(sales[[
    "sale_date",
    "line_activation_date",
    "is_line_activated"
]].head())

,sale_date,line_activation_date,is_line_activated
0,2018-06-29,2018-06-29,1
1,2018-06-30,2018-07-01,1
2,2018-06-30,2018-07-01,1
3,2018-06-30,2018-07-01,1
4,2018-06-30,NaT,0


In [ ]:
#Calculate days to activation show how many days between sale date and the activation date
sales["days_to_activation"] = (
    sales["line_activation_date"] - sales["sale_date"]
).dt.days

display(sales[[
    "sale_date",
    "line_activation_date",
    "days_to_activation"
]].head())

,sale_date,line_activation_date,days_to_activation
0,2018-06-29,2018-06-29,0.0
1,2018-06-30,2018-07-01,1.0
2,2018-06-30,2018-07-01,1.0
3,2018-06-30,2018-07-01,1.0
4,2018-06-30,NaT,NaN


In [ ]:
#Create sales count column easy kpi later
sales["sales_count"] = 1

In [ ]:
#Check sales categories
print("Sales type:")
print(sales["sales_type"].value_counts(dropna=False))

print("\nLine type:")
print(sales["line_type"].value_counts(dropna=False))

print("\nActivation flag:")
print(sales["is_line_activated"].value_counts(dropna=False))

#Final check of Sales table
print("Final Sales table shape:", sales.shape)

print("\nColumns:")
print(sales.columns.tolist())

print("\nMissing values:")
display(sales.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(sales.head())

Sales type:
sales_type
SMS888    9305
SNOC       290
Name: count, dtype: int64

Line type:
line_type
4G     7797
3G     1797
NaN       1
Name: count, dtype: int64

Activation flag:
is_line_activated
1    9160
0     435
Name: count, dtype: int64
Final Sales table shape: (9595, 9)

Columns:
['customer_msisdn', 'sale_date', 'line_activation_date', 'sales_type', 'line_type', 'pdv_subno', 'is_line_activated', 'days_to_activation', 'sales_count']

Missing values:


,missing_count
customer_msisdn,0
sale_date,0
line_activation_date,435
sales_type,0
line_type,1
pdv_subno,0
is_line_activated,0
days_to_activation,435
sales_count,0



Preview:


,customer_msisdn,sale_date,line_activation_date,sales_type,line_type,pdv_subno,is_line_activated,days_to_activation,sales_count
0,846931477,2018-06-29,2018-06-29,SMS888,4G,869107410,1,0.0,1
1,846939913,2018-06-30,2018-07-01,SMS888,4G,867110788,1,1.0,1
2,846854401,2018-06-30,2018-07-01,SMS888,3G,869116085,1,1.0,1
3,846841682,2018-06-30,2018-07-01,SMS888,4G,867114282,1,1.0,1
4,846797123,2018-06-30,NaT,SMS888,4G,869112791,0,NaN,1


In [ ]:
#fact sales
fact_sales = sales[[
    "customer_msisdn",
    "sale_date",
    "line_activation_date",
    "sales_type",
    "line_type",
    "pdv_subno",
    "is_line_activated",
    "days_to_activation",
    "sales_count"
]].copy()

print("fact_sales shape:", fact_sales.shape)
display(fact_sales.head())

fact_sales shape: (9595, 9)


,customer_msisdn,sale_date,line_activation_date,sales_type,line_type,pdv_subno,is_line_activated,days_to_activation,sales_count
0,846931477,2018-06-29,2018-06-29,SMS888,4G,869107410,1,0.0,1
1,846939913,2018-06-30,2018-07-01,SMS888,4G,867110788,1,1.0,1
2,846854401,2018-06-30,2018-07-01,SMS888,3G,869116085,1,1.0,1
3,846841682,2018-06-30,2018-07-01,SMS888,4G,867114282,1,1.0,1
4,846797123,2018-06-30,NaT,SMS888,4G,869112791,0,NaN,1


# Clean the Refill table

In [ ]:
#basic clean
refill = standardize_table(refill_raw)

print("Refill table after basic cleaning:")
print(refill.shape)
print(refill.columns.tolist())

display(refill.head())
#rename column
refill = refill.rename(columns={
    "refill_date": "refill_date",
    "msisdn": "customer_msisdn",
    "msisdn_src": "source_msisdn",
    "refill_amnt": "refill_amount",
    "rfl_src": "refill_source",
    "rfl_chnl": "refill_channel",
    "rfl_acnt": "refill_account",
    "rfl_type": "refill_type",
    "rfl_prp": "refill_purpose",
    "city": "refill_city",
    "prvc": "refill_wilaya"
})

print(refill.columns.tolist())
display(refill.head())

Refill table after basic cleaning:
(9481, 11)
['refill_date', 'msisdn', 'msisdn_src', 'refill_amnt', 'rfl_src', 'rfl_chnl', 'rfl_acnt', 'rfl_type', 'rfl_prp', 'city', 'prvc']


,refill_date,msisdn,msisdn_src,refill_amnt,rfl_src,rfl_chnl,rfl_acnt,rfl_type,rfl_prp,city,prvc
0,19/03/2024,849425285,848715686,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,AIN EL HADJAR,SAIDA
1,19/03/2024,861894177,870203358,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,BOUATI MAHMOUD,GUELMA
2,19/03/2024,852294110,864708211,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CONSTANTINE,CONSTANTINE
3,19/03/2024,842274369,870268752,150,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,ORAN,ORAN
4,19/03/2024,843825102,848709760,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CHETTIA,CHLEF


['refill_date', 'customer_msisdn', 'source_msisdn', 'refill_amount', 'refill_source', 'refill_channel', 'refill_account', 'refill_type', 'refill_purpose', 'refill_city', 'refill_wilaya']


,refill_date,customer_msisdn,source_msisdn,refill_amount,refill_source,refill_channel,refill_account,refill_type,refill_purpose,refill_city,refill_wilaya
0,19/03/2024,849425285,848715686,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,AIN EL HADJAR,SAIDA
1,19/03/2024,861894177,870203358,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,BOUATI MAHMOUD,GUELMA
2,19/03/2024,852294110,864708211,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CONSTANTINE,CONSTANTINE
3,19/03/2024,842274369,870268752,150,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,ORAN,ORAN
4,19/03/2024,843825102,848709760,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CHETTIA,CHLEF


In [ ]:
#clean id same kima les autre
refill["customer_msisdn"] = refill["customer_msisdn"].apply(clean_id)
refill["source_msisdn"] = refill["source_msisdn"].apply(clean_id)

display(refill[["customer_msisdn", "source_msisdn"]].head())

,customer_msisdn,source_msisdn
0,849425285,848715686
1,861894177,870203358
2,852294110,864708211
3,842274369,870268752
4,843825102,848709760


In [ ]:
#Convert date and amount
refill["refill_date"] = parse_date(refill["refill_date"])
refill["refill_amount"] = clean_numeric(refill["refill_amount"])

display(refill[["refill_date", "refill_amount"]].head())

print(refill[["refill_date", "refill_amount"]].dtypes)

,refill_date,refill_amount
0,2024-03-19,500
1,2024-03-19,100
2,2024-03-19,500
3,2024-03-19,150
4,2024-03-19,100


refill_date      datetime64[ns]
refill_amount             int64
dtype: object


In [ ]:
#Convert date and amount
refill["refill_date"] = parse_date(refill["refill_date"])
refill["refill_amount"] = clean_numeric(refill["refill_amount"])

display(refill[["refill_date", "refill_amount"]].head())

print(refill[["refill_date", "refill_amount"]].dtypes)

,refill_date,refill_amount
0,2024-03-19,500
1,2024-03-19,100
2,2024-03-19,500
3,2024-03-19,150
4,2024-03-19,100


refill_date      datetime64[ns]
refill_amount             int64
dtype: object


In [ ]:
#Remove rows with essential missing values because when we saay a row is importatnt if it contains refill date customer MSISDN refill amount and if it miss others it still important if it has this
print("Before removing invalid refill rows:", refill.shape)

refill = refill.dropna(subset=[
    "refill_date",
    "customer_msisdn",
    "refill_amount"
])

print("After removing invalid refill rows:", refill.shape)

Before removing invalid refill rows: (9481, 11)
After removing invalid refill rows: (9481, 11)


In [ ]:
#Remove invalid refill amounts it should be +
print("Before amount filtering:", refill.shape)

refill = refill[refill["refill_amount"] > 0]

print("After amount filtering:", refill.shape)

Before amount filtering: (9481, 11)
After amount filtering: (9481, 11)


In [ ]:
#Create refill count column for more eaasy in power bi
refill["refill_count"] = 1

In [ ]:
#Check refill categories
print("Refill source:")
print(refill["refill_source"].value_counts(dropna=False))

print("\nRefill channel:")
print(refill["refill_channel"].value_counts(dropna=False))

print("\nRefill account:")
print(refill["refill_account"].value_counts(dropna=False))

print("\nRefill type:")
print(refill["refill_type"].value_counts(dropna=False))

print("\nRefill purpose:")
print(refill["refill_purpose"].value_counts(dropna=False))

Refill source:
refill_source
IPOS    9481
Name: count, dtype: int64

Refill channel:
refill_channel
USSD    8906
SNOC     575
Name: count, dtype: int64

Refill account:
refill_account
MAIN ACCOUNT         9195
DEDICATED ACCOUNT     286
Name: count, dtype: int64

Refill type:
refill_type
TOP-UP          9314
BILL PAYMENT     167
Name: count, dtype: int64

Refill purpose:
refill_purpose
FLEXY      6723
ADD-ONS    2758
Name: count, dtype: int64


In [ ]:
#test reffil amount staatic
refill["refill_amount"].describe()

,refill_amount
count,9481.000000
mean,451.404071
std,599.674567
min,50.000000
25%,100.000000
50%,150.000000
75%,500.000000
max,10000.000000


In [ ]:
#test final
print("Final Refill table shape:", refill.shape)

print("\nColumns:")
print(refill.columns.tolist())

print("\nMissing values:")
display(refill.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(refill.head())

Final Refill table shape: (9481, 12)

Columns:
['refill_date', 'customer_msisdn', 'source_msisdn', 'refill_amount', 'refill_source', 'refill_channel', 'refill_account', 'refill_type', 'refill_purpose', 'refill_city', 'refill_wilaya', 'refill_count']

Missing values:


,missing_count
refill_date,0
customer_msisdn,0
source_msisdn,0
refill_amount,0
refill_source,0
refill_channel,0
refill_account,0
refill_type,0
refill_purpose,0
refill_city,0



Preview:


,refill_date,customer_msisdn,source_msisdn,refill_amount,refill_source,refill_channel,refill_account,refill_type,refill_purpose,refill_city,refill_wilaya,refill_count
0,2024-03-19,849425285,848715686,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,AIN EL HADJAR,SAIDA,1
1,2024-03-19,861894177,870203358,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,BOUATI MAHMOUD,GUELMA,1
2,2024-03-19,852294110,864708211,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CONSTANTINE,CONSTANTINE,1
3,2024-03-19,842274369,870268752,150,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,ORAN,ORAN,1
4,2024-03-19,843825102,848709760,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CHETTIA,CHLEF,1


In [ ]:
#Create fact_refill
fact_refill = refill[[
    "refill_date",
    "customer_msisdn",
    "source_msisdn",
    "refill_amount",
    "refill_source",
    "refill_channel",
    "refill_account",
    "refill_type",
    "refill_purpose",
    "refill_city",
    "refill_wilaya",
    "refill_count"
]].copy()

print("fact_refill shape:", fact_refill.shape)
display(fact_refill.head())

fact_refill shape: (9481, 12)


,refill_date,customer_msisdn,source_msisdn,refill_amount,refill_source,refill_channel,refill_account,refill_type,refill_purpose,refill_city,refill_wilaya,refill_count
0,2024-03-19,849425285,848715686,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,AIN EL HADJAR,SAIDA,1
1,2024-03-19,861894177,870203358,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,BOUATI MAHMOUD,GUELMA,1
2,2024-03-19,852294110,864708211,500,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CONSTANTINE,CONSTANTINE,1
3,2024-03-19,842274369,870268752,150,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,ORAN,ORAN,1
4,2024-03-19,843825102,848709760,100,IPOS,USSD,MAIN ACCOUNT,TOP-UP,FLEXY,CHETTIA,CHLEF,1


# Clean the Activations table

In [ ]:
#basic cleaning
activations = standardize_table(activations_raw)

print("Activations table after basic cleaning:")
print(activations.shape)
print(activations.columns.tolist())

display(activations.head())

#Rename columns
activations = activations.rename(columns={
    "subscribernumber": "customer_msisdn",
    "adjustmentdate": "activation_transaction_date",
    "channel_code": "activation_channel_code",
    "transactioncode": "package_code",
    "adjustmentamount": "activation_amount"
})

print(activations.columns.tolist())
display(activations.head())

Activations table after basic cleaning:
(10000, 5)
['subscribernumber', 'adjustmentdate', 'channel_code', 'transactioncode', 'adjustmentamount']


,subscribernumber,adjustmentdate,channel_code,transactioncode,adjustmentamount
0,865811048,18/07/2018,4,INTSPEEDDAY1PRE,100
1,861988207,02/07/2018,4,LIBERTYDAY50,50
2,865224119,19/07/2018,4,INTAMIGO1PRE,30
3,863807917,15/07/2018,1,IMTIYAZ50PRE,50
4,851649153,19/08/2018,4,INTBSPEEDDAY1PRE,100


['customer_msisdn', 'activation_transaction_date', 'activation_channel_code', 'package_code', 'activation_amount']


,customer_msisdn,activation_transaction_date,activation_channel_code,package_code,activation_amount
0,865811048,18/07/2018,4,INTSPEEDDAY1PRE,100
1,861988207,02/07/2018,4,LIBERTYDAY50,50
2,865224119,19/07/2018,4,INTAMIGO1PRE,30
3,863807917,15/07/2018,1,IMTIYAZ50PRE,50
4,851649153,19/08/2018,4,INTBSPEEDDAY1PRE,100


In [ ]:
#Clean ID columns
activations["customer_msisdn"] = activations["customer_msisdn"].apply(clean_id)
activations["activation_channel_code"] = activations["activation_channel_code"].apply(clean_id)
activations["package_code"] = activations["package_code"].apply(clean_id)

display(activations[[
    "customer_msisdn",
    "activation_channel_code",
    "package_code"
]].head())

,customer_msisdn,activation_channel_code,package_code
0,865811048,4,INTSPEEDDAY1PRE
1,861988207,4,LIBERTYDAY50
2,865224119,4,INTAMIGO1PRE
3,863807917,1,IMTIYAZ50PRE
4,851649153,4,INTBSPEEDDAY1PRE


In [ ]:
#Convert date and amount
activations["activation_transaction_date"] = parse_date(
    activations["activation_transaction_date"]
)

activations["activation_amount"] = clean_numeric(
    activations["activation_amount"]
)

display(activations[[
    "activation_transaction_date",
    "activation_amount"
]].head())

print(activations[[
    "activation_transaction_date",
    "activation_amount"
]].dtypes)

,activation_transaction_date,activation_amount
0,2018-07-18,100
1,2018-07-02,50
2,2018-07-19,30
3,2018-07-15,50
4,2018-08-19,100


activation_transaction_date    datetime64[ns]
activation_amount                       int64
dtype: object


In [ ]:
#Remove rows with essential missing values customer date package code
print("Before removing invalid activation rows:", activations.shape)

activations = activations.dropna(subset=[
    "customer_msisdn",
    "activation_transaction_date",
    "package_code"
])

print("After removing invalid activation rows:", activations.shape)

Before removing invalid activation rows: (10000, 5)
After removing invalid activation rows: (10000, 5)


In [ ]:
#Check activation amount values so we know
print("Missing activation amount:")
print(activations["activation_amount"].isna().sum())

print("\nActivation amount statistics:")
display(activations["activation_amount"].describe())

print("\nTop highest activation amounts:")
display(activations.sort_values("activation_amount", ascending=False).head(10))

Missing activation amount:
0

Activation amount statistics:


,activation_amount
count,10000.000000
mean,289.448500
std,477.275098
min,0.000000
25%,50.000000
50%,100.000000
75%,150.000000
max,2500.000000



Top highest activation amounts:


,customer_msisdn,activation_transaction_date,activation_channel_code,package_code,activation_amount
6299,870528521,2022-07-26,1,PREPAIDDJEZZYINTERNET2500,2500
9921,852112959,2024-03-15,2,MIXTEPRE2500,2500
9659,852252125,2024-01-31,2,MIXTEPRE2500,2500
9641,842994368,2024-02-16,1,MIXTEPRE2500,2500
6581,843187626,2023-01-04,1,NEWHAYLABEZZEF2000,2000
6577,870510994,2022-07-21,1,NEWHAYLABEZZEF2000,2000
6575,841774040,2022-04-07,4,B2CSP1PREP2000,2000
6522,862810598,2021-12-30,1,HAYLABEZZEFMONTH2000PREP,2000
8669,843094231,2023-05-11,2,MIXTEPRE2000,2000
3294,841023527,2019-10-08,4,HAYLABEZZEFMONTH2000PREP,2000


In [ ]:
#Create activation transaction count (Each row represents one activation/adjustment transaction.)
activations["activation_transaction_count"] = 1

In [ ]:
#Check activation channels and package codes This helps us see which channels are used most which packages are activated most
print("Activation channel codes:")
print(activations["activation_channel_code"].value_counts(dropna=False).head(20))

print("\nTop package codes:")
print(activations["package_code"].value_counts(dropna=False).head(20))

Activation channel codes:
activation_channel_code
4         5981
1         1765
27         859
29         332
49         260
12         194
2          127
53         106
48          74
18          60
NaN         56
IVR777      28
11          27
52          27
35          24
21          18
45          18
19          17
40          14
25           8
Name: count, dtype: int64

Top package codes:
package_code
LIBERTYDAY50                1083
HAYLABEZZEFDAY100PREP        600
MAXIHAYLA100                 428
GIFTWALKWIN2GO               417
HADRADAY50PREP               394
LIBERTYDAY100                391
HAYLABEZZEFMONTH1500PREP     355
DOVINTSPEEDDAY100MOPRE       321
INTAMIGO1DATADEFAULTPRE      270
MIXTEPRE100                  248
BTLINTSPEEDDAY2GO            235
DOVINTSPEEDDAY1GOPRE         228
GOAHDER                      217
DOVINTSPEEDDAY250MOPRE       186
TRANQUILO                    178
LOWVALUE50                   164
HADRADAY100PREP              153
MIXTEPRE1000                 1

In [ ]:
#Check relation with Package table to know idaa fact_activations[package_code] can conncet dim_package[package_code] later in power bi
activation_package_codes = set(activations["package_code"].dropna())
package_codes = set(dim_package["package_code"].dropna())

matched_codes = activation_package_codes.intersection(package_codes)

match_rate = len(matched_codes) / len(activation_package_codes) * 100

print("Unique package codes in activations:", len(activation_package_codes))
print("Unique package codes in dim_package:", len(package_codes))
print("Matched package codes:", len(matched_codes))
print("Match rate:", round(match_rate, 2), "%")


Unique package codes in activations: 212
Unique package codes in dim_package: 955
Matched package codes: 202
Match rate: 95.28 %


In [ ]:
#Show unmatched package codes so we know why and incase in the cleaning incomplet
unmatched_codes = activation_package_codes - package_codes

unmatched_package_codes = pd.DataFrame({
    "unmatched_package_code": list(unmatched_codes)
})

print("Number of unmatched package codes:", len(unmatched_package_codes))
display(unmatched_package_codes.head(20))


Number of unmatched package codes: 10


,unmatched_package_code
0,CONSULTATION_CREDIT_IVR710
1,EPAYBONUS10DA
2,EPAYBONUS30DA
3,IVR777
4,MIXTEPREPAYG
5,UPSELLHAYLABEZZEFDAILY150PRE
6,EPAYBONUS100DA
7,TRANQUILO
8,EPAYBONUS200DA
9,EPAYBONUS50DA


In [ ]:
#final check
print("Final Activations table shape:", activations.shape)

print("\nColumns:")
print(activations.columns.tolist())

print("\nMissing values:")
display(activations.isna().sum().to_frame("missing_count"))

print("\nPreview:")
display(activations.head())

Final Activations table shape: (10000, 6)

Columns:
['customer_msisdn', 'activation_transaction_date', 'activation_channel_code', 'package_code', 'activation_amount', 'activation_transaction_count']

Missing values:


,missing_count
customer_msisdn,0
activation_transaction_date,0
activation_channel_code,56
package_code,0
activation_amount,0
activation_transaction_count,0



Preview:


,customer_msisdn,activation_transaction_date,activation_channel_code,package_code,activation_amount,activation_transaction_count
0,865811048,2018-07-18,4,INTSPEEDDAY1PRE,100,1
1,861988207,2018-07-02,4,LIBERTYDAY50,50,1
2,865224119,2018-07-19,4,INTAMIGO1PRE,30,1
3,863807917,2018-07-15,1,IMTIYAZ50PRE,50,1
4,851649153,2018-08-19,4,INTBSPEEDDAY1PRE,100,1


In [ ]:
#Create fact_activations
fact_activations = activations[[
    "customer_msisdn",
    "activation_transaction_date",
    "activation_channel_code",
    "package_code",
    "activation_amount",
    "activation_transaction_count"
]].copy()

print("fact_activations shape:", fact_activations.shape)
display(fact_activations.head())

fact_activations shape: (10000, 6)


,customer_msisdn,activation_transaction_date,activation_channel_code,package_code,activation_amount,activation_transaction_count
0,865811048,2018-07-18,4,INTSPEEDDAY1PRE,100,1
1,861988207,2018-07-02,4,LIBERTYDAY50,50,1
2,865224119,2018-07-19,4,INTAMIGO1PRE,30,1
3,863807917,2018-07-15,1,IMTIYAZ50PRE,50,1
4,851649153,2018-08-19,4,INTBSPEEDDAY1PRE,100,1


# export

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data")

RAW_DIR = PROJECT_DIR / "raw_data"
CLEAN_DIR = PROJECT_DIR / "clean_data"
BACKUP_DIR = PROJECT_DIR / "BACKUP_DIR"
REPORT_DIR = PROJECT_DIR / "reports"

print("Project data folder:", PROJECT_DIR)
print("Raw folder:", RAW_DIR)
print("Clean folder:", CLEAN_DIR)
print("Backup folder:", BACKUP_DIR)
print("Reports folder:", REPORT_DIR)

Project data folder: /content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data
Raw folder: /content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data/raw_data
Clean folder: /content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data/clean_data
Backup folder: /content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data/BACKUP_DIR
Reports folder: /content/drive/MyDrive/usthb/PFE_Djezzy/project_code/data/reports


In [ ]:
#backup cleaan befor fact dim
cust.to_csv(BACKUP_DIR / "clean_customers_working.csv", index=False, encoding="utf-8-sig")
pos.to_csv(BACKUP_DIR / "clean_pos_working.csv", index=False, encoding="utf-8-sig")
package.to_csv(BACKUP_DIR / "clean_package_working.csv", index=False, encoding="utf-8-sig")
sales.to_csv(BACKUP_DIR / "clean_sales_working.csv", index=False, encoding="utf-8-sig")
refill.to_csv(BACKUP_DIR / "clean_refill_working.csv", index=False, encoding="utf-8-sig")
activations.to_csv(BACKUP_DIR / "clean_activations_working.csv", index=False, encoding="utf-8-sig")

print("Backup cleaned working tables exported successfully.")

Backup cleaned working tables exported successfully.


In [ ]:
#power bi data file ready
dim_customer.to_csv(CLEAN_DIR / "dim_customer.csv", index=False, encoding="utf-8-sig")
dim_pos.to_csv(CLEAN_DIR / "dim_pos.csv", index=False, encoding="utf-8-sig")
dim_package.to_csv(CLEAN_DIR / "dim_package.csv", index=False, encoding="utf-8-sig")

fact_sales.to_csv(CLEAN_DIR / "fact_sales.csv", index=False, encoding="utf-8-sig")
fact_refill.to_csv(CLEAN_DIR / "fact_refill.csv", index=False, encoding="utf-8-sig")
fact_activations.to_csv(CLEAN_DIR / "fact_activations.csv", index=False, encoding="utf-8-sig")

print("Final Power BI tables exported successfully.")

Final Power BI tables exported successfully.


In [ ]:
#test
print("Backup files:")
for file in BACKUP_DIR.glob("*.csv"):
    print("-", file.name)

print("\nClean Power BI files:")
for file in CLEAN_DIR.glob("*.csv"):
    print("-", file.name)

Backup files:
- clean_customers_working.csv
- clean_pos_working.csv
- clean_package_working.csv
- clean_sales_working.csv
- clean_refill_working.csv
- clean_activations_working.csv

Clean Power BI files:
- dim_customer.csv
- dim_pos.csv
- dim_package.csv
- fact_sales.csv
- fact_refill.csv
- fact_activations.csv
